In [1]:
import os
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import meteostat
from meteostat import Point, monthly
from datetime import datetime
import scipy

## Load data

In [2]:
path = '/Users/trac.k.y/Documents/sta540/cs2/data/' 
all_files = glob.glob(os.path.join(path, "*.csv"))

mens_list = []
womens_list = []

for filename in all_files:
    df = pd.read_csv(filename, header=0)

    if "womens" in filename.lower():
        womens_list.append(df)
    elif "mens" in filename.lower():
        mens_list.append(df)

mens_df = pd.concat(mens_list, axis=0, ignore_index=True)
womens_df = pd.concat(womens_list, axis=0, ignore_index=True)

In [3]:
womens_df[(womens_df["athlete_name"] == "Tracy Sun") & (womens_df["distance"] == "2.74 Miles")] 

,race_date,grade,athlete_name,time,meet_name,distance,is_PR,is_SR,is_improvement
916,Sep 12 2019,11,Tracy Sun,21:42.72,Mt Hamilton Division - Meet #1,2.74 Miles,0,0,0
983,Oct 9 2019,11,Tracy Sun,21:26.22,Mt Hamilton Division - Meet #3,2.74 Miles,0,0,1
1018,Oct 16 2019,11,Tracy Sun,20:56.02,Mt Hamilton Division - Meet #4,2.74 Miles,1,1,1
1091,Aug 29 2018,10,Tracy Sun,22:33.02,LELAND Time Trials,2.74 Miles,0,0,0
1107,Sep 6 2018,10,Tracy Sun,22:19.72,MHAL League Meet #1,2.74 Miles,0,0,1
1137,Sep 13 2018,10,Tracy Sun,22:07.42,MHAL League Meet #2,2.74 Miles,0,0,1
1193,Oct 17 2018,10,Tracy Sun,21:57.52,MHAL League #3,2.74 Miles,0,0,1
1218,Oct 24 2018,10,Tracy Sun,21:38.52,MHAL League Meet #4,2.74 Miles,0,1,1


## Getting the local weather

Question. Does it make more sense to choose the closest weather station with complete temp and humidity data (current method: ex. get both temp and rhum from 1st station. upside: consistent to a station. downside: might end up using farther data than necessary), OR, choose the closest weather stations for both temp and humidity (ex. you could get like temp from closest and if humidity is missing then humidity from second closest), OR, get the mean across all the nearby stations that have any values for temp or humidity (ex. combo of first-third for temp and first-second for humidity)

Question 2. Using 1 model and having gender as fixed effect, OR, separating them? (since I don't particularly care about diff btwn teams and already know they are going to be diff)

Question 3. Clustering by athlete or season? (As mentioned in the proposal comments)
Talk about the new variables created (time_pct_imp)
Also mention that you might want to change that one to quadratic lol even though in SAP you said its linear

Question 4. Add interaction term between temp and humidity (just by testing, seems like interaction is only signfiicant (but also really small) once both are removed. however it probably has to do with scale as well? got a warning about it) ? heteroskedascity in residuals?

In [16]:
race_dates = sorted(list(set(pd.to_datetime(womens_df["race_date"]))))

# point on map 
tpr = Point(37.3020843,-121.7625934) # location of monty hill

stations = meteostat.stations.nearby(tpr, limit=5)
stations

,name,country,region,latitude,longitude,elevation,timezone,distance
id,,,,,,,,
KRHV0,San Jose / Alum Rock,US,CA,37.3329,-121.8198,41,America/Los_Angeles,6110.1
KSJC0,San Jose / Santa Clara Trailer Village,US,CA,37.3627,-121.9291,19,America/Los_Angeles,16191.2
KE160,San Martin,US,CA,37.0816,-121.5968,86,America/Los_Angeles,28578.8
74509,Moffett Field,US,CA,37.4333,-122.0500,10,America/Los_Angeles,29291.5
KPAO0,Palo Alto / Runnymeade (Historical),US,CA,37.4611,-122.1151,2,America/Los_Angeles,35815.2


In [17]:
# point on map 
tpr = Point(37.5033369,-122.3114485) # location of crystal springs

stations = meteostat.stations.nearby(tpr, limit=8)
stations

,name,country,region,latitude,longitude,elevation,timezone,distance
id,,,,,,,,
KSQL0,San Carlos / Silver Penny Mobile Home Park,US,CA,37.5119,-122.2495,2,America/Los_Angeles,5546.7
72494,San Francisco Airport,US,CA,37.6167,-122.3667,3,America/Los_Angeles,13513.5
KHAF0,Half Moon Bay / El Granada Mobile Home Park,US,CA,37.5134,-122.5012,20,America/Los_Angeles,16774.8
KPAO0,Palo Alto / Runnymeade (Historical),US,CA,37.4611,-122.1151,2,America/Los_Angeles,17950.6
72585,Hayward / Russell City,US,CA,37.6589,-122.1218,16,America/Los_Angeles,24052.1
74509,Moffett Field,US,CA,37.4333,-122.0500,10,America/Los_Angeles,24352.7
72493,Metro Oakland International Airport,US,CA,37.7167,-122.2333,2,America/Los_Angeles,24703.4
74506,Alameda Naval Air Station,US,CA,37.7833,-122.3167,4,America/Los_Angeles,31133.9


All distances reported above are in meters.

Notably, although San Francisco Airport and Half Moon Bay are closer in geographic distance to the Crystal Springs course in Belmont, CA, their climate is drastically different enough that I opted to use Hayward's weather station instead if the one in San Carlos was unavailable. See this website:

https://weatherspark.com/compare/s/2/488~145212~149698~545~517~518/Comparison-of-the-Average-Fall-Weather-in-Belmont-San-Francisco-International-Airport-San-Carlos-Airport-Palo-Alto-Half-Moon-Bay-and-Hayward


DELETE IF SUBMIT THIS CODE FOR FINAL IT IS JUST ME RAMBLING

DONE: I can probably get the weather for crystal springs: average from the hours of like 9-3 instead (dumb solution, not all times are on the website and i am only guessing bc sometimes we had invitationals in the morning there and sometimes we had meets). i COULd check if its the weekend and then do 9-1 and then if not do like 3-5 though idk

Also make sure that when ur chekcing if crystal springs, besides specifyig 2.95 also specify the meet_name has to have BVAL or crystal springs in it to be considered (for  example gilroy invitational was also listed as 2.95 miles, https://www.athletic.net/CrossCountry/meet/159089)
ok update i think that litearlly every other invitational except that gilroy one was actually crystal (except for like 1 random toro i think but i lost it) sooooo in lieu of further scraping perhaps just exclude meet name is gilroy and be done with it


BETTER TASK LIST (pseudocode ish)
1) Match the race_dates to distance and meet_name 
2) If distance[idx] == "2.74 Miles" then set point to be monty hill
3) Else if distance[idx] == "2.95 Miles" and meet_name is not gilroy whatever, set point to be crystal springs
    a) And set time to be 9-13 if its a weekend and 13-15 if weekday
4) Else, skip getting weather data
5) Proceed with the weather gathering process

TASK 2 TODO: Find a way to incorporate wind data, by averaging the direction and speed properly. (its some vector shit)

In [ ]:
# 1): match race dates and distance and meet name
womens_df["race_date"] = pd.to_datetime(womens_df["race_date"])
race_dates_comb = womens_df[["race_date", "distance", "meet_name"]].drop_duplicates().sort_values(by="race_date")

weather_saved = dict()

for idx, race_date in enumerate(race_dates_comb["race_date"]):
    row = race_dates_comb.iloc[idx, ]
    race_date = row["race_date"]
    race_day = pd.to_datetime(race_date).tz_localize("America/Los_Angeles") #set timezone as pacific time

    ## 2) set station
    if row["distance"] == "2.74 Miles":
        # point on map 
        tpr = Point(37.3020843,-121.7625934) # location of monty hill
        start_time = 15
        end_time = 17
        start_local = race_day + pd.Timedelta(hours=start_time)
        end_local = race_day + pd.Timedelta(hours=end_time)
        stations = meteostat.stations.nearby(tpr, limit=5)
    elif (row["distance"] == "2.95 Miles") and (row["meet_name"] != "Gilroy Mustang Invitational"):
        tpr = Point(37.5033369,-122.3114485) # location of crystal springs
        ## 3) a) get correct time period for race
        # weekend races are usually 9am-1pm and weekday usually 1-4 pm (more or less)
        if race_day.day_name() in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]:
            start_time = 13
            end_time = 15
        else:
            start_time = 9
            end_time = 13
        start_local = race_day + pd.Timedelta(hours=9)
        end_local = race_day + pd.Timedelta(hours=13)
        stations = meteostat.stations.nearby(tpr, limit=7)
    else:
        continue # skip collecting weather data for other courses bc there are way too many of them and i don't want to deal with them right now
    
    start_utc = start_local.tz_convert("UTC").tz_localize(None)
    end_utc = end_local.tz_convert("UTC").tz_localize(None)
    weather_data = meteostat.hourly(stations, start_utc, end_utc)
    weather_data = weather_data.fetch(units=meteostat.UnitSystem.IMPERIAL)

    for station in stations.index:
        # if the station doesn't have data, or it's SFO/half moon bay: 
        # SF and half moon bay are way cooler than crystal springs on average
        if (station not in weather_data.index.get_level_values(0)) or (station == "72494") or (station == "KHAF0") or (station == "KWVI0"):
            continue 

        station_data = weather_data.loc[station]
        station_data.index = (
            pd.to_datetime(station_data.index)
            .tz_localize("UTC")
            .tz_convert("America/Los_Angeles")
        )

        filtered = station_data[
            (station_data.index.date == race_day.date()) &
            (station_data.index.hour >= start_time) &
            (station_data.index.hour <= end_time)
        ][["temp", "rhum"]]
        # print(filtered)

        if filtered.empty:
            continue
        if filtered[["temp", "rhum"]].isnull().any().any():
            continue
        if not filtered.empty and race_day.date() not in weather_saved:
            weather_mean = filtered.mean()
            weather_mean["station_id"] = station
            weather_saved[race_day.date()] = weather_mean
            break

womens_weather = pd.DataFrame.from_dict(weather_saved, orient="index")

In [7]:
# so its probs stupid to do the guys and girls teams separately since i really can't recall any races that were restricted to 1 of the genders but whatever! just to be rigorous!

# 1): match race dates and distance and meet name
mens_df["race_date"] = pd.to_datetime(mens_df["race_date"])
race_dates_comb = mens_df[["race_date", "distance", "meet_name"]].drop_duplicates().sort_values(by="race_date")

weather_saved = dict()

for idx, race_date in enumerate(race_dates_comb["race_date"]):
    row = race_dates_comb.iloc[idx, ]
    race_date = row["race_date"]
    race_day = pd.to_datetime(race_date).tz_localize("America/Los_Angeles") #set timezone as pacific time

    ## 2) set station
    if row["distance"] == "2.74 Miles":
        # point on map 
        tpr = Point(37.3020843,-121.7625934) # location of monty hill
        start_time = 15
        end_time = 17
        start_local = race_day + pd.Timedelta(hours=start_time)
        end_local = race_day + pd.Timedelta(hours=end_time)
        stations = meteostat.stations.nearby(tpr, limit=5)
    elif (row["distance"] == "2.95 Miles") and (row["meet_name"] != "Gilroy Mustang Invitational"):
        tpr = Point(37.5033369,-122.3114485) # location of crystal springs
        ## 3) a) get correct time period for race
        # weekend races are usually 9am-1pm and weekday usually 1-4 pm (more or less)
        if race_day.day_name() in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]:
            start_time = 13
            end_time = 15
        else:
            start_time = 9
            end_time = 13
        start_local = race_day + pd.Timedelta(hours=9)
        end_local = race_day + pd.Timedelta(hours=13)
        stations = meteostat.stations.nearby(tpr, limit=7)
    else:
        continue # skip collecting weather data for other courses bc there are way too many of them and i don't want to deal with them right now
    
    start_utc = start_local.tz_convert("UTC").tz_localize(None)
    end_utc = end_local.tz_convert("UTC").tz_localize(None)
    weather_data = meteostat.hourly(stations, start_utc, end_utc)
    weather_data = weather_data.fetch(units=meteostat.UnitSystem.IMPERIAL)

    for station in stations.index:
        # if the station doesn't have data, or it's SFO/half moon bay: 
        # SF and half moon bay are way cooler than crystal springs on average
        # https://weatherspark.com/compare/s/2/488~145212~149698~545~517~518/Comparison-of-the-Average-Fall-Weather-in-Belmont-San-Francisco-International-Airport-San-Carlos-Airport-Palo-Alto-Half-Moon-Bay-and-Hayward
        if (station not in weather_data.index.get_level_values(0)) or (station == "72494") or (station == "KHAF0"):
            continue 

        station_data = weather_data.loc[station]
        station_data.index = (
            pd.to_datetime(station_data.index)
            .tz_localize("UTC")
            .tz_convert("America/Los_Angeles")
        )

        filtered = station_data[
            (station_data.index.date == race_day.date()) &
            (station_data.index.hour >= start_time) &
            (station_data.index.hour <= end_time)
        ][["temp", "rhum"]]
        # print(filtered)
        if filtered.empty:
            continue
        if filtered[["temp", "rhum"]].isnull().any().any():
            continue
        if not filtered.empty and race_day.date() not in weather_saved:
            weather_mean = filtered.mean()
            weather_mean["station_id"] = station
            weather_saved[race_day.date()] = weather_mean
            break

mens_weather = pd.DataFrame.from_dict(weather_saved, orient="index")

In [8]:
# prepare columns for the joining
mens_df["race_date"] = pd.to_datetime(mens_df["race_date"]).dt.normalize()
womens_df["race_date"] = pd.to_datetime(womens_df["race_date"]).dt.normalize()

mens_weather.index = pd.to_datetime(mens_weather.index).normalize()
womens_weather.index = pd.to_datetime(womens_weather.index).normalize()

In [9]:
womens_df = pd.merge(left=womens_df, right=womens_weather,
         left_on="race_date", right_index=True,
         how="left")
mens_df = pd.merge(left=mens_df, right=mens_weather,
         left_on="race_date", right_index=True,
         how="left")

In [10]:
womens_df[(womens_df["athlete_name"] == "Tracy Sun") & (womens_df["distance"] == "2.74 Miles")] 

,race_date,grade,athlete_name,time,meet_name,distance,is_PR,is_SR,is_improvement,temp,rhum,station_id
916,2019-09-12,11,Tracy Sun,21:42.72,Mt Hamilton Division - Meet #1,2.74 Miles,0,0,0,90.666667,25.666667,KSJC0
983,2019-10-09,11,Tracy Sun,21:26.22,Mt Hamilton Division - Meet #3,2.74 Miles,0,0,1,72.666667,34.000000,KSJC0
1018,2019-10-16,11,Tracy Sun,20:56.02,Mt Hamilton Division - Meet #4,2.74 Miles,1,1,1,68.000000,44.333333,KRHV0
1091,2018-08-29,10,Tracy Sun,22:33.02,LELAND Time Trials,2.74 Miles,0,0,0,78.800000,45.000000,KRHV0
1107,2018-09-06,10,Tracy Sun,22:19.72,MHAL League Meet #1,2.74 Miles,0,0,1,77.000000,50.000000,KRHV0
1137,2018-09-13,10,Tracy Sun,22:07.42,MHAL League Meet #2,2.74 Miles,0,0,1,73.400000,41.000000,KRHV0
1193,2018-10-17,10,Tracy Sun,21:57.52,MHAL League #3,2.74 Miles,0,0,1,73.633333,36.000000,KSJC0
1218,2018-10-24,10,Tracy Sun,21:38.52,MHAL League Meet #4,2.74 Miles,0,1,1,73.600000,37.000000,KSJC0


In [11]:
womens_df[(womens_df["athlete_name"] == "Tracy Sun") & (womens_df["distance"] == "2.95 Miles")] 

,race_date,grade,athlete_name,time,meet_name,distance,is_PR,is_SR,is_improvement,temp,rhum,station_id
951,2019-09-21,11,Tracy Sun,21:13.04,Gilroy Mustang Invitational,2.95 Miles,1,1,0,NaN,NaN,NaN
1000,2019-10-12,11,Tracy Sun,22:09.64,Crystal Springs Invitational,2.95 Miles,0,0,0,68.72,34.4,72585
1063,2019-11-04,11,Tracy Sun,24:12.44,BVAL (CCS) Championships,2.95 Miles,0,0,0,69.80,33.0,72585
1165,2018-10-06,10,Tracy Sun,23:14.15,Crystal Springs Invitational Belmont CA,2.95 Miles,0,1,0,68.00,60.0,KSQL0


## Save data

In [12]:
womens_df.to_csv(f"/Users/trac.k.y/Documents/sta540/cs2/final_data/womens_df_final.csv")
mens_df.to_csv(f"/Users/trac.k.y/Documents/sta540/cs2/final_data/mens_df_final.csv")

DELETE IF FOR FINAL

"Are there overall trends/patterns in team speed across years?" (ex. 2017 slow -> 2018-2019 fast -> 2020-2022 slow)

"What factors are related to overall speed of team?"
- size of team
- grade distribution of team
- amount of improvement (in terms of time) over course of year / total career
- weather - probably infeasible since i would have to do it manually for each race :/
- retention vs churn?????? (like how often are runners from previous years retained? / alternatively, percentage of new runners per year AND it matters what their speed is ex. 2019 a bunch of fast freshmen joined which would be diff if for example a bunch of jv reverse girls joined)
    i think what im measuring is retention not churn? we want to retain a lot of runners from prev seasons, but also gain new faster runners if that makes sense

bonus:
predict individual improvement?

i also believe i should just start with monty hill since its 2.74 iconic so i can specifically compare for that course. 3 miles could be toro or anywhere else tbh so those experiences could be diff and not comparable (e.g. the courses could be of diff difficulties and that would impact the improvement and time seen)

Ok based on my machinations I believe crystal springs with its 2.95 miles is also accurate to just crystal springs specifically

In [48]:
womens_df["race_date"] = pd.to_datetime(womens_df["race_date"])
mens_df["race_date"] = pd.to_datetime(mens_df["race_date"])

minutes = womens_df["time"].str.split(":", expand=True).astype(float)
womens_df["time_min"] = minutes[0] + minutes[1] / 60

minutes = mens_df["time"].str.split(":", expand=True).astype(float)
mens_df["time_min"] = minutes[0] + minutes[1] / 60

In [52]:
np.mean(
    womens_df[(womens_df["race_date"].dt.year == 2019) & 
              (womens_df["distance"] == "2.74 Miles")]["time_min"])

np.float64(22.817491633986926)

In [73]:
scipy.stats.ttest_ind(womens_df[
              (womens_df["distance"] == "2.74 Miles") & 
              (womens_df["athlete_name"] == "Cindy Zhao")]["time_min"],
              womens_df[
              (womens_df["distance"] == "2.74 Miles") & 
              (womens_df["athlete_name"] == "Melissa Cichon")]["time_min"])

TtestResult(statistic=np.float64(11.577036715368292), pvalue=np.float64(1.0446551935334346e-06), df=np.float64(9.0))

In [81]:
womens_df[womens_df["distance"] == "2.95 Miles"]["meet_name"].unique()

<StringArray>
[               '2nd Annual Fighting Knights Joust',
                     'Crystal Springs Invitational',
      'BVAL Championships (Crystal Springs Course)',
     '2023 CIF Central Coast Section Championships',
                '1st Annual Fighting Knights Joust',
               'Crystal Springs BVAL Championships',
     '2022 CIF Central Coast Section Championships',
                           'Crystal Springs Invite',
                               'BVAL League Finals',
             'BVAL Championships - Crystal Springs',
                                 'CCS Championship',
                                      'BVAL Finals',
          'CIF-Central Coast Section Championships',
  'Blossom Valley Athletic League XC Championships',
     '2025 CIF Central Coast Section Championships',
                      'Gilroy Mustang Invitational',
                         'BVAL (CCS) Championships',
                                    'CCS XC Finals',
          'Crystal Springs Invit

In [84]:
womens_df[womens_df["distance"] == "2.74 Miles"].shape

(1514, 12)

In [85]:
womens_df[womens_df["distance"] == "2.95 Miles"].shape

(680, 12)

In [83]:
womens_df[(womens_df['distance'].isin(["2.74 Miles", "2.95 Miles"]))].shape

(2194, 12)

In [79]:
womens_df[(~womens_df['distance'].isin(["2.74 Miles", "2.95 Miles"])) & (womens_df["race_date"].dt.year != 2020)]["meet_name"].unique()

<StringArray>
[                  '4th Annual Jackie Henderson Memorial',
                      '2nd Annual Fighting Knights Joust',
                                       'Ram Invitational',
                                'Terry Ward Invitational',
                              'Monterey Bay Invitational',
                      '1st Annual Fighting Knights Joust',
                                 'Artichoke Invitational',
                     'Terry Ward Bellarmine Invitational',
             '2022 CIF State Cross Country Championships',
                                 'Earlybird Invitational',
                                 'Palma Chieftan Classic',
                              'Westmoor Ram Invitational',
           'Monterey Bay Invitational - Saturday Edition',
                'CIF-Central Coast Section Championships',
             '2008 CIF State Cross Country Championships',
                             'Foot Locker West Regionals',
                                    'Lowel